# Marker Attention Analysis — `WideModelAttention`

Extracts the C×C cross-marker attention weights from the `channel_attn` layer and visualises
which markers attend to which — per cell type.

The attention matrix entry `A[i, j]` tells us: *"when processing marker i, how much does the
model look at marker j?"*  Biologically, high values on specific off-diagonal entries suggest
that co-expression of those markers is informative for cell-type identity.

**Sections**
1. Setup & paths  
2. Load model backbone from checkpoint  
3. Load validation patches + cell-type labels  
4. Extract attention weights via forward hook  
5. Aggregate attention per cell type  
6. Heatmaps — global + per cell type  
7. Marker attention ranking per cell type  
8. Marker clustering by attention profile  
9. Differential attention (cell-type-specific)  

## 1 · Setup & paths

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import h5py
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform

sys.path.insert(0, str(Path('..').resolve() / 'src'))
from models import WideModelAttention

# ── Paths — update RUN_DIR to point at any CIMATT run ─────────────────────────
BASE = Path('/home/simon_g/isilon_images_mnt/10_MetaSystems/MetaSystemsData/_simon/src/MCA')

RUN_DIR      = BASE / 'z_RUNS/IMC_NB_FineCT_CIMATT_Norm_VICReg'   # ← change to any CIMATT run
MARKERS_TXT  = BASE / '../data/MCI_data/h5_files/IMC_NB_FineCT/used_markers.txt'
H5_FILE      = BASE / '../data/MCI_data/h5_files/IMC_NB_FineCT/IMC_NB_FineCT.h5'
VAL_IDX_TXT  = BASE / '../data/MCI_data/h5_files/IMC_NB_FineCT/val.txt'
IGNORE_LABEL = 'Other'

PATCH_SIZE   = 24   # must match config
CUTTER_SIZE  = 12   # central crop
N_CELLS      = 2000  # how many val cells to process (increase for more stable estimates)
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'

# Backbone hyperparams — must match the backbone config used for this run
BACKBONE_CFG = dict(
    stem_width   = 32,
    block_width  = 2,
    layer_config = [1, 1],
    drop_prob    = 0.05,
    n_heads      = 4,
    input_norm   = True,   # set False if using the non-norm CIMATT run
)

print(f'Run:    {RUN_DIR.name}')
print(f'Device: {DEVICE}')

## 2 · Load model backbone from checkpoint

In [ ]:
def load_backbone(run_dir: Path, cfg: dict) -> WideModelAttention:
    """Load backbone weights from the last mmengine checkpoint in run_dir."""
    ckpt_path = Path((run_dir / 'last_checkpoint').read_text().strip())
    ckpt      = torch.load(ckpt_path, map_location='cpu')
    state     = ckpt['state_dict']
    bb_state  = {k[len('backbone.'):]: v
                 for k, v in state.items() if k.startswith('backbone.')}

    # in_channels is inferred from the stem weight shape
    n_markers = bb_state['stem.0.weight'].shape[0] // cfg['stem_width']
    backbone  = WideModelAttention(in_channels=n_markers, **cfg)
    backbone.load_state_dict(bb_state, strict=True)
    backbone.eval()
    return backbone


backbone     = load_backbone(RUN_DIR, BACKBONE_CFG).to(DEVICE)
N_MARKERS    = backbone.in_channels
marker_names = np.loadtxt(MARKERS_TXT, dtype=str)
assert len(marker_names) == N_MARKERS, \
    f'Marker list length {len(marker_names)} != model in_channels {N_MARKERS}'

print(f'Backbone loaded  |  {N_MARKERS} markers  |  '
      f'{sum(p.numel() for p in backbone.parameters()):,} params')
print('Markers:', list(marker_names))

## 3 · Load validation patches + cell-type labels

In [ ]:
val_idx = np.loadtxt(VAL_IDX_TXT, dtype=int)
rng     = np.random.default_rng(42)
chosen  = rng.choice(val_idx, size=min(N_CELLS, len(val_idx)), replace=False)
chosen.sort()

half = PATCH_SIZE // 2

patches     = []
cell_labels = []

with h5py.File(H5_FILE, 'r') as h5:
    DIM1      = h5['coords']['DIM1'][chosen]
    DIM2      = h5['coords']['DIM2'][chosen]
    sample_id = h5['coords']['sample_id'][chosen].astype(str)
    labels    = h5['coords']['annotation'][chosen].astype(str)

    for d1, d2, sid, lab in zip(DIM1, DIM2, sample_id, labels):
        if lab == IGNORE_LABEL:
            continue
        img = h5['data'][sid]['image']
        s1  = max(0, d1 - half);  e1 = min(img.shape[0], d1 + half)
        s2  = max(0, d2 - half);  e2 = min(img.shape[1], d2 + half)
        p   = img[s1:e1, s2:e2, :]                                # [H, W, C]
        p   = np.pad(p,
                     ((d1-half - min(0, d1-half), max(0, d1+half - img.shape[0])),
                      (d2-half - min(0, d2-half), max(0, d2+half - img.shape[1])),
                      (0, 0)), mode='constant')
        # central crop to cutter_size
        off = (PATCH_SIZE - CUTTER_SIZE) // 2
        p   = p[off:off+CUTTER_SIZE, off:off+CUTTER_SIZE, :]      # [cs, cs, C]
        patches.append(p)
        cell_labels.append(lab)

X            = torch.from_numpy(np.stack(patches)).permute(0, 3, 1, 2).float()  # [N, C, cs, cs]
cell_labels  = np.array(cell_labels)
cell_types   = sorted(set(cell_labels))

print(f'Loaded {len(X)} cells  |  {len(cell_types)} cell types')
for ct in cell_types:
    print(f'  {ct}: {(cell_labels == ct).sum()}')

## 4 · Extract attention weights via forward hook

We attach a hook to `backbone.channel_attn` (the `nn.MultiheadAttention` module).
Its output is `(attn_out, attn_weights)` where `attn_weights` has shape `[B, C, C]`
(attention weight from every query marker to every key marker, averaged over heads).

In [ ]:
BATCH = 128

all_attn   = []   # will collect [B, C, C] tensors

def _attn_hook(module, inp, out):
    # out = (attn_output [S,B,D], attn_weights [B,C,C])
    if out[1] is not None:
        all_attn.append(out[1].detach().cpu())   # [B, C, C]

hook_handle = backbone.channel_attn.register_forward_hook(_attn_hook)

backbone.eval()
with torch.no_grad():
    for start in range(0, len(X), BATCH):
        batch = X[start:start+BATCH].to(DEVICE)
        backbone(batch)

hook_handle.remove()

# Stack into [N, C, C]
attn_all = torch.cat(all_attn, dim=0).numpy()   # [N, C, C]
print(f'Attention tensor: {attn_all.shape}  (cells × markers × markers)')

## 5 · Aggregate attention per cell type

In [ ]:
# Global mean
attn_global = attn_all.mean(axis=0)   # [C, C]

# Per cell type
attn_per_ct = {}
for ct in cell_types:
    mask = cell_labels == ct
    attn_per_ct[ct] = attn_all[mask].mean(axis=0)   # [C, C]
    print(f'  {ct:<20s}  n={mask.sum()}')

## 6 · Attention heatmaps — global + per cell type

Row = query marker ("I am processing this marker"),  
Column = key marker ("I am looking at this marker"),  
Colour = attention weight → how much the row marker attends to the column marker.

In [ ]:
def attn_heatmap(matrix, title, ax, vmin=None, vmax=None, cmap='viridis'):
    """Plot a C×C attention matrix."""
    sns.heatmap(
        matrix,
        xticklabels=marker_names,
        yticklabels=marker_names,
        ax=ax,
        cmap=cmap,
        square=True,
        vmin=vmin,
        vmax=vmax,
        cbar_kws={'shrink': 0.6, 'label': 'attention weight'},
        linewidths=0,
    )
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.tick_params(axis='x', labelrotation=90, labelsize=6)
    ax.tick_params(axis='y', labelrotation=0,  labelsize=6)
    ax.set_xlabel('Key marker (attended to)', fontsize=8)
    ax.set_ylabel('Query marker (attending)', fontsize=8)


fig, ax = plt.subplots(figsize=(12, 10))
attn_heatmap(attn_global, f'Global mean attention  ({len(X)} cells)', ax)
plt.tight_layout()
plt.savefig(RUN_DIR / 'attn_global.pdf', bbox_inches='tight')
plt.show()

In [ ]:
n_ct  = len(cell_types)
ncols = min(3, n_ct)
nrows = int(np.ceil(n_ct / ncols))

# Shared colour scale across all cell types for fair comparison
vmin = min(m.min() for m in attn_per_ct.values())
vmax = max(m.max() for m in attn_per_ct.values())

fig, axes = plt.subplots(nrows, ncols,
                          figsize=(ncols * 7, nrows * 6))
axes = np.array(axes).flatten()

for i, ct in enumerate(cell_types):
    n = (cell_labels == ct).sum()
    attn_heatmap(attn_per_ct[ct], f'{ct}  (n={n})',
                 axes[i], vmin=vmin, vmax=vmax)

for ax in axes[n_ct:]:
    ax.set_visible(False)

fig.suptitle('Per-cell-type mean attention', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(RUN_DIR / 'attn_per_celltype.pdf', bbox_inches='tight')
plt.show()

## 7 · Marker attention ranking per cell type

Column-sum of the attention matrix = total attention **received** by each marker.  
High values → many other markers look at this marker → it is a key reference point for that cell type.

In [ ]:
# Build a (n_ct × n_markers) matrix of received attention per cell type
received = np.stack(
    [attn_per_ct[ct].sum(axis=0) for ct in cell_types], axis=0
)  # [n_ct, C]

# Sort markers by variance across cell types (most discriminative first)
marker_variance = received.var(axis=0)
order           = np.argsort(marker_variance)[::-1]

fig, ax = plt.subplots(figsize=(max(14, N_MARKERS * 0.4), 5))
x       = np.arange(N_MARKERS)
width   = 0.8 / n_ct
colors  = plt.cm.tab20(np.linspace(0, 1, n_ct))

for i, (ct, col) in enumerate(zip(cell_types, colors)):
    ax.bar(x + i * width,
           received[i][order],
           width=width, label=ct, color=col, alpha=0.85)

ax.set_xticks(x + width * (n_ct - 1) / 2)
ax.set_xticklabels(marker_names[order], rotation=75, ha='right', fontsize=7)
ax.set_ylabel('Summed received attention', fontsize=10)
ax.set_title('Attention received per marker, per cell type\n'
             '(markers sorted by variance across cell types)', fontsize=11)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig(RUN_DIR / 'attn_ranking.pdf', bbox_inches='tight')
plt.show()

## 8 · Marker clustering by attention profile

Hierarchical clustering of markers using the global attention matrix as a similarity measure.
Markers that attend to similar sets of other markers cluster together.
These clusters reveal co-expression modules — biologically related marker groups.

In [ ]:
# Use correlation distance between row profiles of the global attention matrix
# Row i = attention pattern of marker i (what it attends to)
from sklearn.preprocessing import StandardScaler

profiles = attn_global   # [C, C]

# Correlation-based distance
corr     = np.corrcoef(profiles)           # [C, C]
dist     = np.clip(1 - corr, 0, 2)        # convert to distance, clip numerical issues
linkage_mat = linkage(squareform(dist), method='ward')
leaf_order  = leaves_list(linkage_mat)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    attn_global[np.ix_(leaf_order, leaf_order)],
    xticklabels=marker_names[leaf_order],
    yticklabels=marker_names[leaf_order],
    ax=ax,
    cmap='viridis',
    square=True,
    cbar_kws={'shrink': 0.6, 'label': 'attention weight'},
    linewidths=0,
)
ax.set_title('Global attention matrix — markers hierarchically clustered by attention profile',
             fontsize=10, fontweight='bold')
ax.tick_params(axis='x', labelrotation=90, labelsize=6)
ax.tick_params(axis='y', labelrotation=0,  labelsize=6)
plt.tight_layout()
plt.savefig(RUN_DIR / 'attn_clustered.pdf', bbox_inches='tight')
plt.show()

print('Marker order after clustering:')
for i, idx in enumerate(leaf_order):
    print(f'  {i+1:2d}. {marker_names[idx]}')

## 9 · Differential attention — what makes each cell type unique?

For each cell type, subtract the global mean attention and show which marker–marker
interactions are *above average* (positive = specific to this cell type).

This is the most biologically interpretable view: a strong positive entry for
(query=CD3, key=CD8) in T_Cell but not in Tumor confirms the model has learned
that CD3–CD8 co-attention is a T-cell-specific signal.

In [ ]:
ncols = min(3, n_ct)
nrows = int(np.ceil(n_ct / ncols))

fig, axes = plt.subplots(nrows, ncols,
                          figsize=(ncols * 7, nrows * 6))
axes = np.array(axes).flatten()

# Shared symmetric colour scale
diffs = {ct: attn_per_ct[ct] - attn_global for ct in cell_types}
abs_max = max(np.abs(d).max() for d in diffs.values())

for i, ct in enumerate(cell_types):
    n = (cell_labels == ct).sum()
    attn_heatmap(
        diffs[ct],
        f'{ct}  (n={n})  — differential attention',
        axes[i],
        vmin=-abs_max, vmax=abs_max,
        cmap='RdBu_r',
    )

for ax in axes[n_ct:]:
    ax.set_visible(False)

fig.suptitle('Differential attention per cell type  (cell type − global mean)\n'
             'Red = stronger-than-average, Blue = weaker-than-average',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(RUN_DIR / 'attn_differential.pdf', bbox_inches='tight')
plt.show()

## 10 · Top marker pairs per cell type

Rank off-diagonal entries of the differential attention matrix to find the most
cell-type-specific marker–marker interactions.

In [ ]:
TOP_K = 10

for ct in cell_types:
    diff = diffs[ct].copy()
    np.fill_diagonal(diff, 0)   # ignore self-attention

    # Flatten and find top-K positive entries
    flat    = diff.flatten()
    top_idx = np.argsort(flat)[::-1][:TOP_K]
    rows, cols = np.unravel_index(top_idx, diff.shape)

    print(f'\n{ct}  — top {TOP_K} differential attention pairs:')
    print(f'  {"Query marker":<20s}  {"Key marker":<20s}  {"Δattention":>12s}')
    print(f'  {"-"*20}  {"-"*20}  {"-"*12}')
    for r, c, v in zip(rows, cols, flat[top_idx]):
        print(f'  {marker_names[r]:<20s}  {marker_names[c]:<20s}  {v:>+12.5f}')